In [ ]:
# @title 1. Setup — upload gói ZIP + cài dependency
import os, pathlib, subprocess, sys, zipfile

SOURCE = "upload"  # @param ["upload", "github"]
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
REPO = "https://github.com/Datbadboiz11/Day21-ThanTienDat-2A202603023-Track3-Finetuning-Lab.git"
PROJECT = pathlib.Path("/content/lab21_2A202603023")

if not (PROJECT / "requirements.txt").exists():
    if SOURCE == "upload":
        from google.colab import files
        print("Chọn COLAB_INPUT_2A202603023.zip (hoặc ZIP kết quả để tiếp tục).")
        uploaded = files.upload()
        archives = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(archives) != 1:
            raise RuntimeError("Hãy tải lên đúng một file ZIP của dự án.")
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                destination = (pathlib.Path("/content") / member.filename).resolve()
                if not destination.is_relative_to(PROJECT) or "\\" in member.filename:
                    raise RuntimeError("ZIP có cấu trúc không đúng; dùng gói do export_colab.py tạo.")
            archive.extractall("/content")
    else:
        subprocess.run(["git", "clone", REPO, str(PROJECT)], check=True)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
for name in ("results", "adapters"):
    (PROJECT / name).mkdir(exist_ok=True)
required = ["data/train_seed.jsonl", "data/eval_target.jsonl", "data/eval_regression.jsonl", "data/checksums.json"]
missing = [name for name in required if not (PROJECT / name).is_file()]
if missing:
    raise RuntimeError(f"Gói ZIP thiếu dữ liệu: {missing}")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
os.environ["COMPUTE_TIER"] = "T4"
os.environ["EPOCHS"] = "2"
os.environ["MASK_MODE"] = "assistant-only"
for name in ("EVAL_LIMIT", "BASE_MODEL", "ONLY", "FORCE_RETRAIN", "LAB_BACKUP_DIR"):
    os.environ.pop(name, None)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chọn Runtime > Change runtime type > T4 GPU, rồi chạy lại ô Setup.")
print("GPU:", torch.cuda.get_device_name(0))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["LAB_BACKUP_DIR"] = "/content/drive/MyDrive/Lab21_2A202603023"
    print("Sao lưu sau mỗi NB và mỗi adapter vào", os.environ["LAB_BACKUP_DIR"])

def run_stages(*stages):
    subprocess.run([sys.executable, "-u", "scripts/colab_run.py", *stages], check=True)


# NB6 (tuỳ chọn) — Merge, kiểm chứng sau merge, và hoán đổi adapter

Deck §23. Hai đường triển khai:
* **Merge** — `W = W₀ + (α/r)·BA`, đồ thị phục vụ giống hệt base → **không** overhead.
* **Giữ riêng** — một base trong VRAM, nhiều adapter, chọn theo từng request.

> Ngày 20 sở hữu tầng phục vụ (vLLM/SGLang). Ở đây chỉ làm phần **quyết định lúc huấn
> luyện ảnh hưởng gì tới lúc phục vụ** — rồi bàn giao.


In [ ]:
import json, os, pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from labkit import evaluate as ev, generate, report
from labkit.config import get_tier

ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "data").exists() else pathlib.Path.cwd().parent
TIER = get_tier(os.environ.get("COMPUTE_TIER", "T4"))

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

# EVAL_LIMIT=0 (unset) means the FULL set here, exactly as it does in NB2 and NB5.
# It used to default to 20, so an unabridged run silently scored this notebook's
# no-regression assert on 20 of 50 items while every other notebook used all of them.
EVAL_LIMIT = int(os.environ.get("EVAL_LIMIT", "0") or 0)
target = load_jsonl(ROOT / "data" / "eval_target.jsonl")
if EVAL_LIMIT:
    target = target[:EVAL_LIMIT]
print(f"scoring merge check on {len(target)} target items")


## 1. Điểm TRƯỚC merge


In [ ]:
from peft import PeftModel

model, tok = generate.load_base(TIER)
model = PeftModel.from_pretrained(model, str(ROOT / "adapters" / "correct"))
preds, _ = generate.generate_batch(model, tok, [r["input"] for r in target],
                                   system=generate.NAIVE_PROMPT)
before = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds, target)) / len(target)
print(f"trước merge: {before:.4f}")


## 2. Merge và đo lại

**Assert bắt buộc.** Merge là phép toán chính xác về mặt lý thuyết, nhưng dtype khi
gộp có thể làm tụt điểm. Nếu tụt, đừng deploy — hãy tìm hiểu vì sao.


In [ ]:
merged = model.merge_and_unload()
preds_m, _ = generate.generate_batch(merged, tok, [r["input"] for r in target],
                                     system=generate.NAIVE_PROMPT)
after = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds_m, target)) / len(target)
delta = after - before
print(f"sau merge:   {after:.4f}   (Δ {delta:+.4f})")

TOL = 0.01
assert delta >= -TOL, (
    f"điểm TỤT {abs(delta):.4f} sau merge (ngưỡng {TOL}). Kiểm tra dtype lúc merge; "
    "với DoRA cần PEFT ≥ 0.10 để gộp đúng vector magnitude (deck §23)."
)

out = ROOT / "adapters" / "merged"
merged.save_pretrained(out); tok.save_pretrained(out)
report.write_json({"before_merge": before, "after_merge": after, "delta": delta,
                   "tolerance": TOL, "n": len(target)},
                  "merge_check.json", results_dir=ROOT / "results")
del merged; generate.free_memory()


## 3. Một base, nhiều adapter — hoán đổi theo request

Đây là lập luận kinh tế của LoRA ở deck §23: base nằm trong VRAM một lần, mỗi khách
hàng/tác vụ là một adapter vài chục MB.


In [ ]:
model, tok = generate.load_base(TIER)
model = PeftModel.from_pretrained(model, str(ROOT / "adapters" / "correct"),
                                  adapter_name="correct")
available = ["correct"]
for extra in ("attn_only", "qlora"):
    d = ROOT / "adapters" / extra
    if d.exists():
        model.load_adapter(str(d), adapter_name=extra)
        available.append(extra)

print("adapter đang nạp:", available)
ticket = target[0]["input"]
for name in available:
    model.set_adapter(name)
    out, _ = generate.generate_batch(model, tok, [ticket], system=generate.NAIVE_PROMPT)
    print(f"\n[{name}] -> {out[0][:140]}")


## ✅ Checkpoint NB6
- [ ] `results/merge_check.json` — điểm sau merge không tụt quá 0.01
- [ ] Đã hoán đổi ≥2 adapter trên **cùng một** base đang nạp

→ Ngày 20 (Model Serving) là nơi biến việc này thành một endpoint đa người thuê.
